In [1]:
import pandas as pd
data = pd.read_csv(r"C:\Users\Admin\Downloads\IoT Dataset\CCIOT\train.csv")

In [4]:
print(data["network_ips_src"].iloc[100])

['192.168.1.193', '192.168.1.19', '192.168.1.1']


In [ ]:
import pandas as pd
data_train = pd.read_csv(r"train.csv")
data_valid = pd.read_csv(r"valid.csv")
data_test = pd.read_csv(r"test.csv")

In [ ]:
import pandas as pd
import ast

def preprocess_dataframe(df):
    print("Bước 0: Tiền xử lý dữ liệu (Parsing & Sorting)...")
    
    # 1. Parse các cột List từ String sang Set (Tập hợp)
    list_columns = ['network_ips_src', 'network_ips_dst', 'network_ports_src', 'network_ports_dst']
    
    def safe_parse_to_set(val):
        try:
            if isinstance(val, str):
                parsed = ast.literal_eval(val)
                return set(parsed) if isinstance(parsed, list) else set()
            return set(val) if isinstance(val, list) else set()
        except (ValueError, SyntaxError):
            return set()

    for col in list_columns:
        print(f"  - Đang parse cột {col}...")
        df[f'{col}_set'] = df[col].apply(safe_parse_to_set)

    # 2. Chuyển đổi Timestamp chuẩn ISO 8601 sang số giây (Unix Epoch Time Float)
    print("  - Đang chuyển đổi Timestamp...")
    # format='ISO8601' giúp pandas phân tích chuỗi T...Z cực nhanh
    df['datetime_obj'] = pd.to_datetime(df['timestamp'], format='ISO8601', utc=True)
    
    # Lấy ra số giây dưới dạng float (chính xác đến microsecond)
    df['timestamp_num'] = df['datetime_obj'].astype('int64') / 1e9

    # 3. SẮP XẾP TĂNG DẦN THEO THỜI GIAN (Bắt buộc cho Sliding Window)
    df = df.sort_values(by='timestamp_num').reset_index(drop=True)
    
    return df

In [ ]:
data_train = preprocess_dataframe(data_train)
data_valid = preprocess_dataframe(data_valid)
data_test = preprocess_dataframe(data_test)

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
from torch_geometric.data import Data 
from collections import deque
from tqdm.notebook import tqdm
import os

# Bật cảnh báo trùng lặp của KMP (Intel MKL)
os.environ["KMP_DUPLICATE_BLOCK_WARNING"] = "TRUE"

def build_ip_topology_graph(df, train_mask, valid_mask, test_mask, window_size=10, max_dt=30.0):
    print(f"Bước 1: Trích xuất Node Features (X) và Label (y)...")
    
    # Các cột không đưa vào Node Feature
    cols_to_drop = [
        "timestamp", "timestamp_num", "network_ips_src", "network_ips_dst", "network_ports_src", "network_ports_dst", "label"
    ]
    
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    # Tạo tensor đặc trưng
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
        
    features_np = np.ascontiguousarray(features_np)
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    print(f"Bước 2: Xây Dựng Đồ Thị Flow-based bằng Sliding Window (Max {max_dt}s)...")

    # Lấy mảng numpy để truy xuất O(1)
    timestamps = df['timestamp_num'].values
    ips_src = df['network_ips_src_set'].values
    ips_dst = df['network_ips_dst_set'].values
    ports_src = df['network_ports_src_set'].values
    ports_dst = df['network_ports_dst_set'].values

    all_src, all_dst, all_edge_attrs = [], [], []
    
    # Hàng đợi hai đầu lưu trữ các Node Index trong cửa sổ thời gian
    window = deque()

    # Hàm tính Jaccard Similarity nhanh
    def jaccard_sim(set_a, set_b):
        if not set_a and not set_b: return 0.0 # Cả 2 đều rỗng
        intersect = len(set_a.intersection(set_b))
        union = len(set_a.union(set_b))
        return intersect / union if union > 0 else 0.0

    for curr_idx in tqdm(range(len(df)), desc="Quét cửa sổ thời gian"):
        curr_time = timestamps[curr_idx]
        curr_ip_src = ips_src[curr_idx]
        curr_ip_dst = ips_dst[curr_idx]
        
        # 1. Loại bỏ các node đã quá cũ khỏi cửa sổ (vượt quá max_dt = 30s)
        while window and (curr_time - timestamps[window[0]]) > max_dt:
            window.popleft()
            
        # 2. Kiểm tra nối cạnh với các node còn lại trong cửa sổ (tối đa window_size node)
        # Ép deque về list và lấy K node cuối cùng (ưu tiên các node sát thời gian nhất)
        recent_nodes = list(window)[-window_size:] 
        
        for past_idx in recent_nodes:
            past_ip_src = ips_src[past_idx]
            past_ip_dst = ips_dst[past_idx]
            
            # ĐIỀU KIỆN NỐI: Tập IP Tổng của curr và past có giao thoa > 0
            curr_ip_all = curr_ip_src.union(curr_ip_dst)
            past_ip_all = past_ip_src.union(past_ip_dst)
            
            if len(curr_ip_all.intersection(past_ip_all)) > 0:
                # Tính dt
                dt_raw = abs(curr_time - timestamps[past_idx])
                # Normalize dt_raw (Log scale + Min-Max [0,1])
                # max_dt là 30s. log1p(30 * 1e6) ~ 17.2 -> Chia cho 18 để an toàn
                dt = np.log1p(dt_raw * 1e6) / 18.0
                
                # Tính Jaccard Overlap
                w_ip_src = jaccard_sim(curr_ip_src, past_ip_src)
                w_ip_dst = jaccard_sim(curr_ip_dst, past_ip_dst)
                
                curr_port_src = ports_src[curr_idx]
                past_port_src = ports_src[past_idx]
                w_port_src = jaccard_sim(curr_port_src, past_port_src)
                
                curr_port_dst = ports_dst[curr_idx]
                past_port_dst = ports_dst[past_idx]
                w_port_dst = jaccard_sim(curr_port_dst, past_port_dst)
                
                # Tạo vector trọng số 5 chiều
                attr = [dt, w_ip_src, w_ip_dst, w_port_src, w_port_dst]
                
                # ĐỒ THỊ CÓ HƯỚNG: Quá khứ trỏ tới Hiện tại
                all_src.append(past_idx)
                all_dst.append(curr_idx)
                all_edge_attrs.append(attr)
                
        # 3. Thêm node hiện tại vào cửa sổ
        window.append(curr_idx)

    print("\nHợp nhất và tạo Cạnh (Culling Edge Index)...")
    # edge_index: tensor 2 x num_edges
    edge_index = torch.tensor([all_src, all_dst], dtype=torch.long).contiguous()
    # edge_attr: tensor num_edges x 5
    edge_attr = torch.tensor(all_edge_attrs, dtype=torch.float32).contiguous()

    print(f"HOÀN THIỆN ĐỒ THỊ TEMPORAL FLOW (CÓ EDGE FEATURES)!")
    print(f"   - Số Nodes : {x_tensor.size(0):,}")
    print(f"   - Số Edges : {edge_index.size(1):,}")
    print(f"   - Số Features/Cạnh : {edge_attr.shape[1]}")
    
    # Dọn rác bộ nhớ RAM
    del features_np, all_src, all_dst, all_edge_attrs, timestamps, ips_src, ips_dst, ports_src, ports_dst
    gc.collect()
    
    # Tạo đối tượng Graph của PyTorch Geometric
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    
    # Gắn mask
    graph.train_mask = train_mask
    graph.valid_mask = valid_mask
    graph.test_mask = test_mask
    
    return graph

In [ ]:
import gc
from torch_geometric.loader import NeighborLoader
import torch
import numpy as np
import pandas as pd

print("=== BƯỚC 1: GỘP DỮ LIỆU ĐÃ TIỀN XỬ LÝ VÀ CHUẨN BỊ MASKS ===")

# 1. Đánh dấu tập dữ liệu
data_train['split_tag'] = 0
data_valid['split_tag'] = 1
data_test['split_tag'] = 2

# 2. Gộp 3 DataFrames
print("Concat thành Siêu Dataframe...")
df_all = pd.concat([data_train, data_valid, data_test], ignore_index=True)

# Giải phóng bộ nhớ ngay lập tức
del data_train, data_valid, data_test
gc.collect()

# 3. Sắp xếp lại toàn cục theo trình tự thời gian
# Dù tập gốc đã sort, khi gộp lại ta VẪN PHẢI sort toàn cục để Sliding Window không bị lỗi ở ranh giới các tập
print("Sắp xếp toàn cục theo trình tự thời gian (Chronological Sort)...")
df_all.sort_values(by='timestamp_num', inplace=True)
df_all.reset_index(drop=True, inplace=True)
gc.collect()

total_nodes = len(df_all)

# 4. Tái tạo Masks động
print("Tái tạo Cấu trúc Masks...")
train_mask = torch.tensor((df_all['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((df_all['split_tag'] == 1).values, dtype=torch.bool)
test_mask  = torch.tensor((df_all['split_tag'] == 2).values, dtype=torch.bool)

df_all.drop(columns=['split_tag'], inplace=True)
gc.collect()

print(f" - Tổng Node: {total_nodes:,}")
print(f" - Train Mask : {train_mask.sum().item():,} ({train_mask.float().mean()*100:.1f}%)")
print(f" - Valid Mask : {valid_mask.sum().item():,} ({valid_mask.float().mean()*100:.1f}%)")
print(f" - Test Mask  : {test_mask.sum().item():,} ({test_mask.float().mean()*100:.1f}%)")

# 5. Build đồ thị
print("\n=== BẮT ĐẦU QUÁ TRÌNH BUILD FLOW-BASED GRAPH ===")
super_graph_faiss = build_ip_topology_graph(
    df_all, 
    train_mask, 
    valid_mask, 
    test_mask, 
    window_size=10, 
    max_dt=30.0
)
print("\nĐồ thị tổng Đã Gắn Masks thành công!")

# 6. Khởi tạo NeighborLoader
print("\nSet up Graph NeighborLoaders...")
train_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
test_loader_faiss  = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.test_mask,  batch_size=2048, shuffle=False, num_workers=0)

del df_all 
gc.collect()
print("Hoàn tất Data Prep cho Graph. Đã sẵn sàng train GAT + XGBoost!")

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
import gc

# định nghĩa gat với 2 lớp
class GAT_Embedder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=5):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training) # chỉ dropout trong quá trình train, bỏ ngẫu nhiên 25% node features
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.25, training=self.training) # chỉ dropout trong quá trình train, bỏ ngẫu nhiên 25% node features
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=7):
    from sklearn.metrics import f1_score
    import numpy as np

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3) 
    
    # learning rate giảm một nửa nếu sau 2 epoch liên tiếp mà F1 validation không cải thiện 
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2, verbose=True
    )
    
    # nếu có class_weights thì truyền vào để xử lý imbalanced dataset
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        
    best_val_f1 = -1.0
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_nodes = 0
        all_train_preds = []
        all_train_labels = []
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
            batch_size = batch.batch_size
            
            # chỉ lấy 20248 embedding của 2028 node gốc để tính loss và đánh giá, bỏ qua hàng xóm hop 1, hop 2
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0) # dùng gradient clipping tránh exploding gradients
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            y_true = batch.y[:batch_size]
            
            all_train_preds.append(pred.cpu().numpy())
            all_train_labels.append(y_true.cpu().numpy())
            
            total_train_loss += loss.item() * batch_size
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
            
            del batch, out, emb, loss, pred, y_true
            
        avg_train_loss = total_train_loss / total_train_nodes
        train_f1 = f1_score(np.concatenate(all_train_labels), np.concatenate(all_train_preds), average='macro')
        
        pbar_train.close()
            
        model.eval()
        total_val_loss = 0
        total_val_nodes = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                
                out, emb = model(batch.x, batch.edge_index, batch.edge_attr)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                y_true = batch.y[:batch_size]
                
                all_val_preds.append(pred.cpu().numpy())
                all_val_labels.append(y_true.cpu().numpy())
                
                total_val_loss += loss.item() * batch_size
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}"})
                
                del batch, out, emb, loss, pred, y_true
                
        pbar_val.close()
                
        avg_val_loss = total_val_loss / total_val_nodes
        val_f1 = f1_score(np.concatenate(all_val_labels), np.concatenate(all_val_preds), average='macro')
        
        scheduler.step(val_f1)
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Macro F1: {train_f1:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Macro F1: {val_f1:.4f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate hiện tại: {current_lr:.6f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve_epochs = 0
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Macro F1: {best_val_f1:.4f} | Val Loss: {avg_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện F1 {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\nĐã kích hoạt Early Stopping tại Epoch {epoch+1}! Lùi về checkpoint tốt nhất.")
                break
                
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nHoàn tất Train! Đã load lại weights MANG LẠI F1 CAO NHẤT (Val Macro F1: {best_val_f1:.4f}) để dùng trích xuất Embedding!")
        
    return model

In [ ]:
# HÀM TRÍCH XUẤT EMBEDDING
from sklearn.metrics import accuracy_score, f1_score, classification_report

@torch.no_grad()
def extract_embeddings_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings...")
    
    # [ĐÃ SỬA]: Đồng bộ num_neighbors=[10, 5] với tập Train
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[10, 5], 
        input_nodes=mask,
        batch_size=512,
        shuffle=False,        
        num_workers=0 
    )
    
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        del batch, emb, _
        
    pbar.close()
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

# HÀM TRAIN XGBOOST
def train_evaluate_xgboost(X_train_emb, y_train, X_valid_emb, y_valid, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # [ĐÃ SỬA]: Xóa hoàn toàn các dòng hardcode class weight từ dataset cũ.
    # Chỉ giữ lại thuật toán làm mượt (sqrt) để không phạt quá nặng các lớp đa số.

    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000, 
        learning_rate=0.08,
        max_depth=5, 
        min_child_weight=2,
        gamma=1.0,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.75,         
        colsample_bytree=0.7,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",  
        early_stopping_rounds=40
    )
    
    print("Đang dọn dẹp RAM & VRAM...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost...")
    try:
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost trên CUDA, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ TRÊN TẬP TEST MASK % ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [ ]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm
import torch
import gc
from torch_geometric.loader import NeighborLoader

# tương tự như hàm trên nhưng sẽ trả về ma trận nối giữa raw features và embedding GAT để train XGBoost với cả 2 loại đặc trưng
@torch.no_grad()
def extract_concat_features_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...")
    
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[10, 5],
        input_nodes=mask,
        batch_size=512,
        shuffle=False,
        num_workers=0 
    )
    
    all_combined = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Nối")
    for batch in pbar:
        batch = batch.to(device)
        
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        bs = batch.batch_size
        
        # lấy raw features và embedding các node gốc
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        
        # concat raw_features và embeddings 
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        
        all_combined.append(combined_matrix)
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        del batch, emb, _, raw_features, embeddings, combined_matrix
        
    pbar.close()
    

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    final_combined = np.concatenate(all_combined, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Nối: {final_combined.shape}")
    return final_combined, final_labels

def train_evaluate_concat_xgboost(X_train, y_train, X_valid, y_valid, X_test, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # [ĐÃ SỬA]: Xóa hoàn toàn các dòng hardcode class weight từ dataset cũ.
    # Chỉ giữ lại thuật toán làm mượt (sqrt) để không phạt quá nặng các lớp đa số.

    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=7, 
        min_child_weight=2,
        gamma=0.4,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.9,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=40
    )
    
    print(" Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)")
    try:
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang Train trên:", device)

print("=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT ===")
import gc
gc.collect()

num_features = super_graph_faiss.x.shape[1] 

# lấy số classs
num_classes = len(torch.unique(super_graph_faiss.y)) 
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# khởi tạo model GAT
model_faiss = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=64,      
     embedding_dim=32,        
     num_classes=num_classes, 
     heads=8,
     edge_dropout=0.3,
     edge_dim=5 
).to(device)

# tạo mảng Class_Weights dựa trên tập train
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_y_cpu = super_graph_faiss.y[super_graph_faiss.train_mask].cpu().numpy()
unique_classes_arr = np.unique(train_y_cpu)
raw_weights_arr = compute_class_weight('balanced', classes=unique_classes_arr, y=train_y_cpu)

# căn bậc 2 để làm mượt trọng số
smoothed_weights = np.sqrt(raw_weights_arr)
# np.clip: giới hạn giá trị của trọng số trong khoảng [0.1,10.0]
smoothed_weights = np.clip(smoothed_weights, a_min=0.1, a_max=10.0)

# chuyển thành tensor và đưa lên device
gat_class_weights_faiss = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)
print(f"Trọng số cân bằng Class Weights GAT: {gat_class_weights_faiss}")

print("\nBắt đầu quá trình Huấn Luyện GAT!")
# chạy hàm train_gat
model_faiss = train_gat(
    model_faiss, 
    train_loader_faiss, 
    valid_loader_faiss, 
    device, 
    epochs=20, 
    lr=0.0005, 
    class_weights=gat_class_weights_faiss,
    patience=6 
)

import os
os.makedirs("model_final", exist_ok=True)
torch.save(model_faiss.state_dict(), "model_final/gat_embedder.pth")
print("\nThành công! Đã lưu trọng số GAT tĩnh vào thư mục 'model_final/gat_embedder.pth'")

In [ ]:
import gc
import os
import torch

print("=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST ===")

# Trích xuất sang Vector cho XGBoost xử lý
print("\n1. Trích xuất Vector cho Train Mask")
X_train_temporal, y_train_temporal = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)

print("\n2. Trích xuất Vector cho Valid Mask")
X_valid_temporal, y_valid_temporal = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)

print("\n3. Trích xuất Vector cho Test Mask")
X_test_temporal, y_test_temporal = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

# [QUAN TRỌNG] Giải phóng hoàn toàn Đồ thị PyTorch Geometric khỏi RAM nhường sân cho XGBoost
print("\nGiải phóng bộ nhớ Đồ thị...")
del super_graph_faiss 
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Huấn luyện và đánh giá XGBoost từ embedding của GAT
print("\n4. Khởi chạy Pipeline Huấn luyện XGBoost")
hybrid_xgb_temporal = train_evaluate_xgboost(
    X_train_temporal, y_train_temporal, 
    X_valid_temporal, y_valid_temporal, 
    X_test_temporal, y_test_temporal
)

# Lưu model XGBoost đã huấn luyện
os.makedirs("model_final", exist_ok=True)
hybrid_xgb_temporal.save_model("model_final/GAT_XGB_Hybrid_Temporal_Model.json")
print("\nThành công! Hybrid XGBoost Temporal Model đã được lưu vào 'model_final/GAT_XGB_Hybrid_Temporal_Model.json'")

In [ ]:
# đánh giá GAT trực tiếp trên tập Test Mask (Không qua XGBoost)
@torch.no_grad()
def evaluate_gat_directly(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...")
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[20, 10],
        input_nodes=mask,
        batch_size=16384,
        shuffle=False,
        num_workers=0 
    )
    all_preds = []
    all_labels = []
    pbar = tqdm(loader, desc="Đang đánh giá trực tiếp GAT")
    for batch in pbar:  
        batch = batch.to(device)
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
        bs = batch.batch_size
        all_preds.append(out[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        del batch, out
    pbar.close()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    final_preds = np.concatenate(all_preds, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)   
    pred_classes = final_preds.argmax(axis=1)
    acc = accuracy_score(final_labels, pred_classes)
    f1_macro = f1_score(final_labels, pred_classes, average='macro')
    f1_weighted = f1_score(final_labels, pred_classes, average='weighted')
    print(f"GAT Đánh giá trực tiếp trên Test Mask:")
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(final_labels, pred_classes, digits=4))
evaluate_gat_directly(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)